In [ ]:
from langchain_chroma import Chroma #to create a vector store and store the embeddings in it, and to query the vector store for relevant documents based on a query
from langchain_community.document_loaders import TextLoader #Text for langchain to work with
from langchain_text_splitters import CharacterTextSplitter #split that document into smaller pieces and meaningful
import torch.nn as nn
import sentence_transformers

C:\Users\Abhinav\AppData\Local\Temp\ipykernel_36600\1801989400.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader #Text for langchain to work with


In [2]:
from dotenv import load_dotenv
load_dotenv() #used for writing into the .env file and loading it for security reasons

True

In [3]:
import pandas as pd
books=pd.read_csv("books_cleaned.csv")
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title and subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883: A NOVEL THAT READERS and critic...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982: A new 'Christie for Christmas' ...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736: A memorable, mesmerizing heroin..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897: Lewis' work on the nature of lo...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934: ""In The Problem of Pain, C.S. L..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222: On A Train Journey Home To Nort...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014: This book tells the tale of a m...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623: Wisdom to Create a Life of Pass...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535: This collection of the timeless...


In [4]:
books["tagged_description"]

0       9780002005883: A NOVEL THAT READERS and critic...
1       9780002261982: A new 'Christie for Christmas' ...
2       9780006178736: A memorable, mesmerizing heroin...
3       9780006280897: Lewis' work on the nature of lo...
4       9780006280934: "In The Problem of Pain, C.S. L...
                              ...                        
5192    9788172235222: On A Train Journey Home To Nort...
5193    9788173031014: This book tells the tale of a m...
5194    9788179921623: Wisdom to Create a Life of Pass...
5195    9788185300535: This collection of the timeless...
5196    9789027712059: Since the three volume edition ...
Name: tagged_description, Length: 5197, dtype: str

In [5]:
books["tagged_description"].to_csv("tagged_description.txt", index=False, header=False,encoding="utf-8")

In [6]:
raw_documents = TextLoader("tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=1, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1171, which is longer than the specified 1
Created a chunk of size 1217, which is longer than the specified 1
Created a chunk of size 376, which is longer than the specified 1
Created a chunk of size 312, which is longer than the specified 1
Created a chunk of size 484, which is longer than the specified 1
Created a chunk of size 485, which is longer than the specified 1
Created a chunk of size 963, which is longer than the specified 1
Created a chunk of size 189, which is longer than the specified 1
Created a chunk of size 846, which is longer than the specified 1
Created a chunk of size 297, which is longer than the specified 1
Created a chunk of size 198, which is longer than the specified 1
Created a chunk of size 882, which is longer than the specified 1
Created a chunk of size 1091, which is longer than the specified 1
Created a chunk of size 1192, which is longer than the specified 1
Created a chunk of size 307, which is longer than the specified 1
Create

In [7]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='"9780002005883: A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, 

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db_books = Chroma.from_documents(
    documents,
    embedding=embeddings
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Abhinav\Desktop\book-recommender\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abhinav\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
db_books._collection.count()

5197

In [12]:
query="space exploration"
docs=db_books.similarity_search(query, k=5)
docs

[Document(id='99232da8-265e-4beb-9cf3-1d632586fc14', metadata={'source': 'tagged_description.txt'}, page_content='"9780345397607: Two explorers--one of information and the other of space--face the collapse of galactic civilization, and their only hope is that the explorer of information, Merinda Neskat, can face up to her past and find the answers civilization needs. $150,000 ad/promo. Tour."'),
 Document(id='4a27a469-1a98-4fbc-9d16-6e249a2759b1', metadata={'source': 'tagged_description.txt'}, page_content='"9780345397614: Two explorers--one of information and the other of space--face the collapse of galactic civilization, and their only hope is that the explorer of information, Merinda Neskat, can face up to her past and find the answers civilization needs. Originally titled: Starshield: Sentinels. Reprint."'),
 Document(id='35b3352b-736f-4664-b7bd-4a8e6baa763f', metadata={'source': 'tagged_description.txt'}, page_content='"9781841492704: On a ship without a mission ... No one remembe

In [ ]:
"""books dataframe
    ↓
isbn13 = 9780345397607
    ↓
tagged_description
    ↓
"9780345397607: Two explorers..."
    ↓
Embedding
    ↓
Chroma
    ↓
Similarity Search
    ↓
Document returned
    ↓
Extract ISBN
    ↓
books[books["isbn13"] == ISBN]
    ↓
Original dataframe row
    ↓
Title, Author, Rating, Thumbnail """

In [18]:
books[books["isbn13"]==int(docs[0].page_content.split()[0].strip('":'))]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title and subtitle,tagged_description
1458,9780345397607,0345397606,Starshield Sentinels,Margaret Weis;Tracy Hickman,Fiction,http://books.google.com/books/content?id=0xPEr...,Two explorers--one of information and the othe...,1996.0,3.51,421.0,179.0,Starshield Sentinels,9780345397607: Two explorers--one of informati...


In [ ]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10
)->pd.DataFrame:
    recs=db_books.similarity_search(query, k=50)
    books_list=[]
    for i in recs: 
        books_list.append(int(i.page_content.split()[0].strip('":')))

    return books[books["isbn13"].isin(books_list)].head(top_k) #function for retrieving recommendations based on a query, it performs a similarity search on the vector store and returns the top k recommendations from the original dataframe based on the ISBNs extracted from the retrieved documents

In [22]:
retrieve_semantic_recommendations("space exploration", top_k=5)

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title and subtitle,tagged_description
114,9780060509057,0060509058,Travels,Michael Crichton,Biography & Autobiography,http://books.google.com/books/content?id=QYilZ...,Often I feel I go to some distant region of th...,2002.0,3.95,400.0,6812.0,Travels,9780060509057: Often I feel I go to some dista...
139,9780060541835,0060541830,Congo,Michael Crichton,Fiction,http://books.google.com/books/content?id=Gp-fL...,Armed with the latest gifts of advanced techno...,2003.0,3.58,442.0,140223.0,Congo,9780060541835: Armed with the latest gifts of ...
549,9780131871656,013187165X,Astronomy,Eric Chaisson;Stephen McMillan,Mathematics,http://books.google.com/books/content?id=1O00A...,This introduction to astronomy features an exc...,2006.0,3.85,499.0,153.0,Astronomy: a beginner's guide to the universe,9780131871656: This introduction to astronomy ...
727,9780141011110,0141011114,The Fabric of the Cosmos,Brian Greene,Science,http://books.google.com/books/content?id=dpSqv...,From the bestselling author of The Elegant Uni...,2005.0,4.12,592.0,324.0,"The Fabric of the Cosmos: Space, Time and the ...",9780141011110: From the bestselling author of ...
833,9780142500378,0142500372,Enchantress from the Stars,Sylvia Louise Engdahl,Juvenile Fiction,http://books.google.com/books/content?id=rntJA...,When young Elana unexpectedly joins the team l...,2003.0,3.94,304.0,2031.0,Enchantress from the Stars,9780142500378: When young Elana unexpectedly j...
